# 27B Scale Validation: N=100 Naive Vote +5pp Robustness

Stress test da claim "+5pp naive vote sobre greedy" do `qwen36-27b-selfconsistency`.

## Setup

- Model: Qwen3.6-27B (dense)
- **Seed 42** (SAME as original eval, but N=100 vs N=20)
- Covers positions 0-99 of seed-42 shuffle (all 80 training + 20 original eval = 100 baseline)
- 2 methods only: greedy + naive_vote (probe was null AUROC 0.509, skip)
- Budget: ~20h (100 prompts × 12 min per prompt)

## Decision rule

- `delta >= +4pp`: claim VALIDATED robustly
- `delta [+2pp, +4pp]`: MARGINAL (effect exists but smaller than originally reported)
- `delta [0, +2pp]`: WEAK (claim essentially null)
- `delta < 0`: FAILED

Crash-safe: saves partial every 3 prompts.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/27b_scale_val'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-27B'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_TARGET = 'caiovicentino1/qwen36-27b-selfconsistency'

N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
N_PROMPTS = 100
SEED = 42  # SAME as 27B original eval — extending, not replacing
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print(f'27B scale validation: N={N_PROMPTS}, seed={SEED}, budget ~20h')

## Cell 2 — Load Qwen3.6-27B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Qwen3.6-27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Load 100 prompts seed 42 (positions 0-99)

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(SEED)
random.shuffle(questions)
eval_set = questions[:N_PROMPTS]
print(f'{len(eval_set)} eval prompts (seed {SEED}, positions 0-{N_PROMPTS-1})')
print(f'First 20 match original eval (expected: same accuracy as prior)')

## Cell 4 — Helpers

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
                    r'^\s*\(?([A-J])\)?\s*\.?\s*$']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_greedy(ids):
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_naive(ids, n_traces=N_TRACES, temp=TEMP):
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
                             num_return_sequences=n_traces, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(n_traces)]
    filtered = [a for a in answers if a]
    # Log divergence for diagnostic
    unique = len(set(filtered)) if filtered else 0
    vote = Counter(filtered).most_common(1)[0][0] if filtered else None
    return vote, answers, unique

print('helpers ready')

## Cell 5 — Run N=100 (crash-safe, save every 3)

In [ ]:
results = {'greedy': [], 'naive_vote': []}
diag = []  # per-prompt diagnostic
t0 = time.time()

# Resume logic: if partial exists, load and skip
partial_path = OUT / 'partial.json'
start_idx = 0
if partial_path.exists():
    try:
        with open(partial_path) as f:
            saved = json.load(f)
        results['greedy'] = [(r[0], r[1]) for r in saved.get('greedy', [])]
        results['naive_vote'] = [(r[0], r[1]) for r in saved.get('naive_vote', [])]
        diag = saved.get('diag', [])
        start_idx = len(results['greedy'])
        print(f'Resuming from position {start_idx}')
    except Exception as e:
        print(f'No valid partial, starting fresh: {e}')

for qi in tqdm(range(start_idx, len(eval_set)), desc='27B N=100', initial=start_idx, total=len(eval_set)):
    q = eval_set[qi]
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536:
        results['greedy'].append((None, False))
        results['naive_vote'].append((None, False))
        diag.append({'qi': qi, 'skipped': 'prompt too long'})
        continue
    gold = q['gold_letter']

    try:
        g = gen_greedy(ids)
        results['greedy'].append((g, g == gold))
    except Exception as e:
        print(f'greedy err qi={qi}: {e}')
        results['greedy'].append((None, False))
        g = None

    try:
        vote, answers, unique = gen_naive(ids)
        results['naive_vote'].append((vote, vote == gold))
        diag.append({'qi': qi, 'gold': gold, 'greedy': g, 'naive': vote,
                     'answers': answers, 'unique_answers': unique})
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        results['naive_vote'].append((None, False))
        diag.append({'qi': qi, 'gold': gold, 'greedy': g, 'naive': None, 'skipped': 'OOM'})
    except Exception as e:
        print(f'naive err qi={qi}: {e}')
        results['naive_vote'].append((None, False))

    if (qi+1) % 3 == 0:
        g_acc = np.mean([r[1] for r in results['greedy']])
        n_acc = np.mean([r[1] for r in results['naive_vote']])
        delta = (n_acc - g_acc) * 100
        n_div = sum(1 for d in diag if d.get('unique_answers', 0) >= 2)
        print(f'  [{qi+1}/{len(eval_set)}]  greedy={g_acc:.3f}  naive={n_acc:.3f}  Δ={delta:+.1f}pp  div={n_div}/{len(diag)}')
        # Crash-safe save
        with open(partial_path, 'w') as f:
            json.dump({
                'greedy': [(r[0], bool(r[1])) for r in results['greedy']],
                'naive_vote': [(r[0], bool(r[1])) for r in results['naive_vote']],
                'diag': diag,
                'current_idx': qi+1,
            }, f, indent=2)

g_acc = np.mean([r[1] for r in results['greedy']])
n_acc = np.mean([r[1] for r in results['naive_vote']])
delta_pp = (n_acc - g_acc) * 100

print(f'\n{"="*60}')
print(f'  FINAL 27B N={len(eval_set)} seed={SEED} ({(time.time()-t0)/3600:.1f}h)')
print(f'{"="*60}')
print(f'  greedy:        {g_acc:.3f}  ({sum(r[1] for r in results["greedy"])}/{len(results["greedy"])})')
print(f'  naive vote:    {n_acc:.3f}  ({sum(r[1] for r in results["naive_vote"])}/{len(results["naive_vote"])})')
print(f'  delta:         {delta_pp:+.1f}pp')

if delta_pp >= 4: verdict = '✅ VALIDATED ROBUSTLY'
elif delta_pp >= 2: verdict = '🟡 MARGINAL'
elif delta_pp >= 0: verdict = '⚠️ WEAK'
else: verdict = '❌ FAILED'
print(f'  verdict:       {verdict}')
print(f'{"="*60}')

# Divergence breakdown
n_div = sum(1 for d in diag if d.get('unique_answers', 0) >= 2)
n_conv = sum(1 for d in diag if d.get('unique_answers', 0) == 1)
print(f'\n=== DIVERGENCE BREAKDOWN ===')
print(f'divergent (unique>=2): {n_div}/{len(diag)}')
print(f'convergent (unique=1): {n_conv}/{len(diag)}')
for subset, flag in [('divergent', True), ('convergent', False)]:
    mask_idxs = [i for i, d in enumerate(diag)
                 if (d.get('unique_answers', 0) >= 2) == flag and 'skipped' not in d]
    if not mask_idxs: continue
    g_sub = np.mean([results['greedy'][i][1] for i in mask_idxs])
    n_sub = np.mean([results['naive_vote'][i][1] for i in mask_idxs])
    print(f'  [{subset}] n={len(mask_idxs)}  greedy={g_sub:.3f}  naive={n_sub:.3f}  Δ={(n_sub-g_sub)*100:+.1f}pp')

## Cell 6 — Upload results

In [ ]:
summary = {
    'model': MODEL_ID,
    'experiment': '27B_scale_validation_n100_seed42',
    'n_prompts': len(eval_set),
    'seed': SEED,
    'n_traces': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'results': {
        'greedy': float(g_acc),
        'naive_vote': float(n_acc),
    },
    'delta_pp': float(delta_pp),
    'verdict': verdict.split()[-1] if '✅' in verdict else verdict.split()[-1],
    'n_divergent': n_div,
    'n_convergent': n_conv,
    'original_n20_comparison': {
        'greedy': 0.550, 'naive_vote': 0.600, 'delta_pp': 5.0
    },
    'eval_date': time.strftime('%Y-%m-%d'),
}
with open(OUT / 'scale_validation_n100.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open(OUT / 'scale_diag.json', 'w') as f:
    json.dump(diag, f, indent=2)

api = HfApi()
for fn in ['scale_validation_n100.json', 'scale_diag.json']:
    api.upload_file(path_or_fileobj=str(OUT/fn), path_in_repo=fn,
                    repo_id=HF_TARGET, repo_type='model',
                    commit_message=f'27B scale validation N={N_PROMPTS} seed=42: delta={delta_pp:+.1f}pp')
    print(f'OK {fn}')

print(f'\n✅ https://huggingface.co/{HF_TARGET}')